# Investment Research Agent
**AAI-520 Final Project — University of San Diego**

**Team:** Viviana Garzon Silva · Obinna Edeh · Dina Othman  
**GitHub repository:** [vgsilva-eng/Natural-Language-Processing](https://github.com/vgsilva-eng/Natural-Language-Processing)

## Project Overview

This project examines how an agentic system can organize a multi-step equity research task while keeping its intermediate decisions visible. Given a stock symbol, the system develops a research plan, selects relevant specialists, gathers current market and news data, produces an analysis, critiques that analysis, and revises it. The AAPL run shown here is an execution example rather than an investment recommendation.

### Agent Design and Workflows

The design separates data retrieval, specialist analysis, and orchestration. This separation makes failures easier to locate: a provider error can be distinguished from a routing error or an unsupported model conclusion. It also keeps deterministic calculations, such as moving averages, outside the language model. The three workflow patterns follow established distinctions among prompt chaining, routing, and evaluator-optimizer systems [1].

| Component | Responsibility |
|---|---|
| **MarketAgent** | Computes price, return, moving-average, and volume signals from Yahoo Finance data. |
| **EarningsAgent** | Organizes revenue, income, margin, earnings-per-share, and valuation measures. |
| **NewsAgent** | Classifies recent articles and summarizes themes that may matter to investors. |
| **ResearchAgent** | Plans the analysis, selects specialists, synthesizes their findings, reflects, revises, and saves memory. |
| **Prompt chaining** | Passes news through ingest, preprocess, classify, extract, and summarize stages. |
| **Routing** | Classifies an input and dispatches it to the corresponding specialist. |
| **Evaluator-optimizer** | Scores a draft against explicit criteria and uses the feedback in a bounded revision loop. |

### Agent Functions and Capabilities

The ResearchAgent addresses the four required agent functions directly:

1. **Planning:** It generates ordered research steps for the requested ticker.
2. **Dynamic tool use:** It maps the plan to the relevant market, earnings, and news specialists instead of relying on a fixed execution sequence.
3. **Self-reflection:** It identifies strengths and omissions in the initial report, then uses that critique to produce a revised analysis.
4. **Learning across runs:** It stores a short ticker-specific record and reloads that record as context during a later run.

Specialist results are returned as structured dictionaries. This choice supports testing and inspection before the information is converted into narrative text.

## 0. Reproducible Setup

The setup cell identifies the repository root, loads configuration from a local `.env` file, and verifies that the required Anthropic and NewsAPI credentials are available. Secret values are never printed and the `.env` file is excluded from version control. One ticker variable controls the examples, which makes the notebook easy to rerun for another company without changing the workflow code.

The implementation follows standard Python readability conventions and keeps public functions documented in line with PEP 8 guidance [4].

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

TICKER = "AAPL"
print("Setup complete. Repo root:", repo_root)

## 1. Data Tools

The tools provide the evidence layer for the agents. `yfinance` supplies historical prices and company financial fields, while NewsAPI retrieves recent articles by keyword and date [2, 3]. Testing these tools before invoking an agent separates data-access problems from reasoning problems.

**Interpretation note:** The returned values reflect the providers' current records at execution time. Missing fields are handled explicitly because coverage can vary by ticker and date. The notebook does not treat a missing value as zero.

In [ ]:
from tools.yfinance_tool import get_stock_data, get_financials

market = get_stock_data(TICKER)
print(f"Ticker:        {market['ticker']}")
print(f"Current price: ${market['current_price']:,.2f}")
print(f"Change today:  {market['change_pct']:+.2f}%")
print(f"52W range:     ${market['week_52_low']} – ${market['week_52_high']}")
print(f"MA50:          {market['ma_50']}")
print(f"History rows:  {len(market['history'])}")

In [ ]:
fin = get_financials(TICKER)
print(f"Sector:        {fin['sector']}")
print(f"Revenue:       {fin['revenue']}")
print(f"Net income:    {fin['net_income']}")
print(f"P/E ratio:     {fin['pe_ratio']}")

In [ ]:
from tools.news_tool import fetch_news

articles = fetch_news(TICKER, lookback_days=7, max_articles=5)
print(f"Articles found: {len(articles)}\n")
for a in articles:
    print(f"  [{a['source']}] {a['title']}")

## 2. MarketAgent

MarketAgent converts price history into a compact technical snapshot. It calculates the period return, 20-day and 50-day moving averages, recent volume relative to average volume, and the stock's position within its 52-week range. These calculations are deterministic and do not require an LLM.

**What the output shows:** The signals describe recent price behavior; they do not establish causation or predict future returns. Keeping the calculation layer separate allows the team to verify each metric against the underlying time series.

In [ ]:
from agents.market_agent import MarketAgent

market_agent = MarketAgent()
market_result = market_agent.analyze(TICKER, period="3mo")

print(f"Ticker:          {market_result['ticker']} as of {market_result['as_of']}")
print(f"Price:           ${market_result['current_price']:,.2f}")
print(f"Period return:   {market_result['period_return_pct']:+.2f}%")
print(f"Trend:           {market_result['trend']}")
print(f"Crossover:       {market_result['crossover']}")
print(f"Volume ratio:    {market_result['volume_ratio']}x 20-session avg")
print(f"\nSummary: {market_result['summary']}")

In [ ]:
from IPython.display import SVG, display
from tools.market_visualization import save_price_chart

chart_path = save_price_chart(market_result, repo_root / "market_price_chart.svg")
display(SVG(filename=str(chart_path)))

## 3. EarningsAgent

EarningsAgent organizes fundamental data into a consistent result containing revenue, net income, earnings per share, net margin, valuation, and short explanatory signals. Formatting functions preserve `N/A` when Yahoo Finance does not return a field, preventing an unavailable observation from being presented as a measured result.

**What the output shows:** Profitability and valuation should be read together. A strong margin may describe operating performance, while the price-to-earnings ratio reflects how the market currently prices that performance. Neither measure is sufficient on its own.

In [ ]:
from agents.earnings_agent import EarningsAgent

def fmt_currency(v, d=0):
    return f"${v:,.{d}f}" if v is not None else "N/A"

def fmt_number(v, d=2):
    return f"{v:.{d}f}" if v is not None else "N/A"

def fmt_pct(v):
    return f"{v:.2f}%" if v is not None else "N/A"

earnings_agent = EarningsAgent()
earnings_result = earnings_agent.analyze(TICKER)

print(f"Ticker:             {earnings_result['ticker']}")
print(f"Sector:             {earnings_result['sector'] or 'N/A'}")
print(f"Industry:           {earnings_result['industry'] or 'N/A'}")
print(f"Revenue:            {fmt_currency(earnings_result['revenue'])}")
print(f"Net Income:         {fmt_currency(earnings_result['net_income'])}")
print(f"EPS:                {fmt_currency(earnings_result['eps'], 2)}")
print(f"P/E Ratio:          {fmt_number(earnings_result['pe_ratio'])}")
print(f"Forward P/E:        {fmt_number(earnings_result['forward_pe'])}")
print(f"Net Profit Margin:  {fmt_pct(earnings_result['net_margin_pct'])}")
print("\nFinancial Signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 4. NewsAgent

NewsAgent retrieves recent articles, assigns each item to a financial topic, and summarizes the main themes. The summarization prompt is restricted to the supplied titles and descriptions so the output remains tied to retrieved evidence. NewsAPI supports keyword, publication-date, source, language, and sorting controls that make this bounded retrieval possible [3].

**What the output shows:** The categories provide organization rather than verified sentiment labels. Article availability and publisher coverage can shape the summary, so the result should be interpreted as a snapshot of the retrieved sample.

In [ ]:
from agents.news_agent import NewsAgent

news_agent = NewsAgent()
news_articles = news_agent.fetch_and_classify(TICKER, lookback_days=7)

print(f"Articles classified: {len(news_articles)}\n")
for article in news_articles[:5]:
    print(f"  [{article['category']:10s}] {article['title']}")

In [ ]:
news_summary = news_agent.summarize(news_articles)
print("News Summary:\n")
print(news_summary)

In [ ]:
# Package the specialist results for comparison with the final orchestrated report.
specialist_results = {
    "market": market_result,
    "earnings": earnings_result,
    "news": {
        "articles_analyzed": len(news_articles),
        "summary": news_summary,
    },
}

print(f"Market data source:      {market_result['data_source']}")
print(f"Earnings data source:    {earnings_result['data_source']}")
print(f"News articles analyzed:  {len(news_articles)}")
print("Earnings signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 5. Workflow 1: Prompt Chaining

Prompt chaining decomposes the news task into five observable stages: ingest, preprocess, classify, extract, and summarize. Each stage consumes the previous stage's output. This structure narrows the responsibility of each model call and exposes intermediate results that can be checked before synthesis. Anthropic describes prompt chaining as appropriate when a task can be divided into clear, sequential subtasks [1].

The workflow also returns a complete empty-result contract when no articles are available. That fallback keeps downstream notebook cells runnable and makes the absence of evidence explicit.

In [ ]:
from workflows.prompt_chaining import run_prompt_chain

chain_result = run_prompt_chain(TICKER, lookback_days=7)

print(f"Articles ingested:    {chain_result['articles_ingested']}")
print(f"After deduplication:  {chain_result['articles_after_dedup']}")
print("\nKey facts:")
for fact in chain_result['key_facts']:
    print(f"  - {fact}")

In [ ]:
print("News Summary:\n")
print(chain_result['summary'])

## 6. Workflow 2: Routing and Dispatch

Routing separates classification from execution. The router first labels an input as `earnings`, `news`, or `market`; the dispatcher then instantiates the matching specialist and returns its structured result. This design gives each specialist a narrower task and makes the selected path visible [1].

The examples below test all three route categories, followed by a dispatch demonstration. Showing the returned ticker and signal count confirms that routing caused specialist execution rather than stopping at a classification label.

In [ ]:
import os
import anthropic
from workflows.routing import classify_route, route_to_specialist

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

samples = [
    "Apple reported quarterly EPS of $1.64, beating estimates by $0.08. Revenue grew 5% YoY.",
    "Analyst upgrades AAPL to Buy, citing strong services momentum.",
    "AAPL is trading above its 50-day moving average on above-average volume.",
]

for content in samples:
    route = classify_route(content, client)
    print(f"Route: {route:10s} | {content[:80]}")

print()
print("Dispatch demonstration:")
dispatch_result = route_to_specialist(samples[2], ticker=TICKER)
print(f"Selected specialist: {dispatch_result['route']}")
print(f"Returned ticker:     {dispatch_result['result'].get('ticker')}")
print(f"Returned signals:    {len(dispatch_result['result'].get('signals', []))}")

## 7. Evaluation and Iteration: Evaluator-Optimizer

The evaluator scores a draft against four stated criteria: numerical specificity, balance between risks and opportunities, clarity of the investment takeaway, and avoidance of vague language. A score below the threshold triggers revision, subject to a maximum of three iterations. The cap prevents an open-ended loop and makes execution cost predictable.

The recorded history matters as much as the final draft because it shows the score, feedback, and stopping decision for each cycle. This follows the evaluator-optimizer pattern in which one model response is assessed and improved through explicit feedback [1]. More rigorous production evaluation would also require a larger set of representative tasks and outcome measures [5].

In [ ]:
from workflows.evaluator_optimizer import run_evaluator_optimizer

eval_result = run_evaluator_optimizer(
    draft=chain_result['summary'],
    threshold=8,
    max_iterations=3
)

print(f"Iterations: {eval_result['iterations']}\n")
for h in eval_result['history']:
    print(f"  Iteration {h['iteration']}: score {h['score']}/10")
    print(f"  Feedback: {h['feedback'][:120]}...\n")

In [ ]:
print("Final refined draft:\n")
print(eval_result['final_draft'])

## 8. ResearchAgent: Integrated Pipeline

ResearchAgent coordinates the complete task. It begins with an LLM-generated plan, scans that plan for market, earnings, and news concepts, and runs only the matching specialists. If the plan is too general to map reliably, the agent falls back to all three specialists so that the analysis does not proceed with an empty evidence base.

The synthesis prompt receives structured market and earnings results, recent headlines, and the NewsAgent's full summary when news analysis is selected. The agent then critiques the initial report and sends both the draft and critique into a second call for revision. This makes reflection operational: the feedback changes the report rather than appearing only as an unused comment.

**Design note:** Keyword-based specialist selection is transparent and easy to test, but it is still a simplified planning policy. A production system would likely use a typed plan schema, stronger validation, and human review for consequential decisions.

In [ ]:
from agents.research_agent import ResearchAgent

agent = ResearchAgent()
report = agent.run(TICKER)

print("Research Plan:")
for i, step in enumerate(report['steps'], 1):
    print(f"  {i}. {step}")

print()
print(f"Specialists selected from plan: {', '.join(report['selected_specialists'])}")

In [ ]:
def display_value(value, currency=False, decimals=2):
    if value is None:
        return "Not selected"
    return f"${value:,.{decimals}f}" if currency else str(value)

print("Market Snapshot:")
snap = report['market_snapshot']
print(f"  Price:     {display_value(snap['price'], currency=True)}")
print(f"  Change:    {display_value(snap['change_pct'])}%")
print(f"  52W high:  {display_value(snap['week_52_high'], currency=True)}")
print(f"  52W low:   {display_value(snap['week_52_low'], currency=True)}")

print()
print("Financials Snapshot:")
fin = report['financials_snapshot']
print(f"  Revenue:    {fin['revenue']}")
print(f"  Net income: {fin['net_income']}")
print(f"  P/E ratio:  {fin['pe_ratio']}")
print(f"  Sector:     {fin['sector']}")

In [ ]:
print("Initial Analysis:")
print()
print(report['initial_analysis'])

print()
print("Revised Analysis After Reflection:")
print()
print(report['analysis'])

In [ ]:
print("Self-Critique Used for Revision:")
print()
print(report['reflection'])

### Persistent Memory Demonstration

After completing a run, ResearchAgent stores the ticker's latest available price and a short report summary. A newly created agent instance then loads the same record from disk. The equality check below demonstrates persistence across object lifetimes rather than reuse of an in-memory variable.

This memory is intentionally brief. It provides context for future planning without treating an earlier model-generated conclusion as permanent truth. A stronger implementation would timestamp records, retain source references, and define expiration rules for stale financial information.

In [ ]:
saved_memory = agent.memory.load().get(TICKER, {})
new_agent_instance = ResearchAgent()
reloaded_memory = new_agent_instance.memory.load().get(TICKER, {})

print(f"Memory saved for {TICKER}:   {bool(saved_memory)}")
print(f"New instance reloaded it:    {saved_memory == reloaded_memory}")
print(f"Remembered last price:       {reloaded_memory.get('last_price')}")
print(f"Remembered summary preview:  {reloaded_memory.get('summary', '')[:160]}...")

---
## 9. Summary and Conclusions

The executed notebook demonstrates the required agent functions and workflow patterns in one traceable system. The specialists produce structured market, earnings, and news results. Prompt chaining exposes intermediate transformations, routing performs actual dispatch, and evaluator-optimizer records a bounded feedback loop. At the orchestration level, ResearchAgent selects specialists from its plan, incorporates available news analysis, revises its initial report from self-critique, and reloads ticker-specific memory across instances.

### Key Results

- The three specialist agents can run independently and return inspectable results.
- ResearchAgent chooses specialists from the generated plan rather than executing a fixed sequence.
- Routing is demonstrated through both route classification and downstream specialist execution.
- Evaluation history records each score, critique, and iteration.
- Reflection is used to revise the initial research narrative.
- Persistent memory is successfully reloaded by a separate ResearchAgent instance.
- The automated suite contains 19 passing tests covering agents, routing, data handling, summaries, visualization, prompt-chain fallbacks, response parsing, and plan-based specialist selection.

### Limitations

- Results depend on third-party availability, rate limits, coverage, and data quality. The `yfinance` project notes that it is an independent research tool that accesses publicly available Yahoo Finance interfaces [2].
- News results reflect the retrieved sample and may omit relevant reporting outside the API's coverage or query window.
- LLM classification, synthesis, and evaluation can vary across runs. A high evaluator score is evidence that the draft met the prompt's criteria; it is not proof of factual correctness or investment value.
- Technical indicators summarize historical observations and should not be interpreted as causal or predictive evidence.
- Memory may preserve stale or model-generated statements unless records are dated, sourced, and periodically refreshed.
- The system does not model portfolio constraints, transaction costs, taxes, liquidity needs, or an individual's risk tolerance. It is an educational research prototype, not financial advice.

### Recommended Extensions

The next useful improvements would be source-level citations in the generated report, typed planning outputs, response caching, cross-ticker comparison, time-aware memory, historical backtesting, and evaluation against a larger labeled benchmark. These changes would improve reproducibility and make the quality claims easier to test.

## References

1. Anthropic. “Building Effective AI Agents.” Describes prompt chaining, routing, and evaluator-optimizer workflow patterns. [https://www.anthropic.com/engineering/building-effective-agents](https://www.anthropic.com/engineering/building-effective-agents)
2. Aroussi, R. *yfinance Documentation*. Documents historical market data, ticker information, and the project's research-use limitations. [https://ranaroussi.github.io/yfinance/](https://ranaroussi.github.io/yfinance/)
3. NewsAPI. *Documentation*. Describes article search, filtering, sorting, and API authentication. [https://newsapi.org/docs](https://newsapi.org/docs)
4. Van Rossum, G., Warsaw, B., and Coghlan, A. “PEP 8 — Style Guide for Python Code.” Python Enhancement Proposals. [https://peps.python.org/pep-0008/](https://peps.python.org/pep-0008/)
5. Anthropic. “Demystifying Evals for AI Agents.” Discusses representative tasks, trial design, graders, and outcome-focused agent evaluation. [https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)
6. Garzon Silva, V., Edeh, O., and Othman, D. *Natural-Language-Processing: Multi-Agent Financial Analysis System*. GitHub repository. [https://github.com/vgsilva-eng/Natural-Language-Processing](https://github.com/vgsilva-eng/Natural-Language-Processing)